In [ ]:
import numpy as np
import pandas as pd
import os
import time
from tqdm import tqdm, trange
import sys
import matplotlib.pyplot as plt
import pickle
import random
import math

# variaveis
L = 5 # lado do lattice
n_lagartos = L**2 # lagartos que cabem no lattice
estrategias = ['O', 'Y', 'B'] # estratégias possíveis
index_map = {'O': 0, 'Y': 1, 'B': 2}
n_geracoes = 2
n_pop = 1 # número de populações independentes
prob_mutacao = None # probabilidade de mutação a cada geração
valores = [4, 8, 24, 48, 80, 120] # número de vizinhos para cada lagarto
a = float(input("Valor de a: "))
matriz_payoff = np.array([[1, 0.5, 2],
                          [2, 1, a],
                          [0.5, 2, 1]])

In [7]:
class Lagarto:
  def __init__(self, i, j, estrategia, fitness, coord_vizinhos_interacao, estrategia_vizinhos_interacao, coord_vizinhos_aprendizado, estrategia_vizinhos_aprendizado, n_vizinhos_interacao, n_vizinhos_aprendizado):
    self.i = i # linha
    self.j = j # coluna
    self.estrategia = estrategia
    self.fitness = 0 # inicia com 0 de fitness
    self.coord_vizinhos_interacao = [] # lista vazia para adicionar as coordenadas dos vizinhos
    self.estrategia_vizinhos_interacao = [] # lista vazia para adicionar as estratégias dos vizinhos
    self.coord_vizinhos_aprendizado = []
    self.estrategia_vizinhos_aprendizado = []
    self.n_vizinhos_interacao = n_vizinhos_interacao # número de vizinhos que o lagarto efetivamente joga
    self.n_vizinhos_aprendizado = n_vizinhos_aprendizado # número de vizinhos que o lagarto olha para aprender

  def calcular_coord_vizinhos(self, L, tipo):
    if tipo == 'interacao':
        n = self.n_vizinhos_interacao
        if n <= 0:
            print("Erro: n_vizinhos_interacao <= 0")
            return

        i0, j0 = self.i, self.j

        def moore(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if not (dx == 0 and dy == 0)
            }

        def von_neumann(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if (dx != 0 or dy != 0) and (abs(dx) + abs(dy) <= r)
            }
        
        r = int(np.ceil((np.sqrt(1 + n) - 1) / 2))

        vn_r = von_neumann(r)
        mo_r = moore(r)

        if n <= 2 * r * (r + 1):    # faixa Moore(r-1) -> VN(r)
            base = moore(r - 1) if r > 1 else set()
            pool = list(vn_r - base)
        else:                       # faixa VN(r) -> Moore(r)
            base = vn_r
            pool = list(mo_r - base)

        faltam = n - len(base)
        if faltam <= 0:
            extras = []
        else:
            idx = np.random.choice(len(pool), size=faltam, replace=False)
            extras = [pool[k] for k in idx]

        self.coord_vizinhos_interacao = list(base) + extras

    elif tipo == 'aprendizado':
        n = self.n_vizinhos_aprendizado
        if n <= 0:
            print("Erro: n_vizinhos_aprendizado <= 0")
            return

        i0, j0 = self.i, self.j

        def moore(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if not (dx == 0 and dy == 0)
            }

        def von_neumann(r):
            return {
                ((i0 + dx) % L, (j0 + dy) % L)
                for dx in range(-r, r + 1)
                for dy in range(-r, r + 1)
                if (dx != 0 or dy != 0) and (abs(dx) + abs(dy) <= r)
            }
        
        r = int(np.ceil((np.sqrt(1 + n) - 1) / 2))

        vn_r = von_neumann(r)
        mo_r = moore(r)

        if n <= 2 * r * (r + 1):    # faixa Moore(r-1) -> VN(r)
            base = moore(r - 1) if r > 1 else set()
            pool = list(vn_r - base)
        else:                       # faixa VN(r) -> Moore(r)
            base = vn_r
            pool = list(mo_r - base)

        faltam = n - len(base)
        if faltam <= 0:
            extras = []
        else:
            idx = np.random.choice(len(pool), size=faltam, replace=False)
            extras = [pool[k] for k in idx]

        self.coord_vizinhos_aprendizado = list(base) + extras
    
  def obter_estrategia_vizinhos(self, matriz_posicao):
      self.estrategia_vizinhos_interacao = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_interacao] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição
      self.estrategia_vizinhos_aprendizado = [matriz_posicao[ni, nj] for ni, nj in self.coord_vizinhos_aprendizado] # dadas as coordenadas, obtém a estratégia do lagarto que ocupa aquela posição

  def mutacao(self, prob_mutacao): # função de mutação
    if np.random.rand() < prob_mutacao: # sorteia um valor entre 0 e 1, se for menor que a probabilidade de mutação, o lagarto muda de estratégia
        estrategias_possiveis = [e for e in estrategias if e != self.estrategia] # obtém as estratégias possíveis, exceto a atual
        self.estrategia = np.random.choice(estrategias_possiveis) # escolhe uma nova estratégia aleatoriamente para mutar

  def calcular_fitness(self, matriz_payoff, index_map): # calcula o fitness do lagarto com base na matriz de payoff e nas estratégias dos vizinhos
    payoff_total = 0
    
    #print(f"Calculando fitness do lagarto na posição ({self.i}, {self.j}) com estratégia {self.estrategia} e vizinhos de aprendizado {self.estrategia_vizinhos_aprendizado}")
    for viz in self.estrategia_vizinhos_aprendizado:
        payoff_total += matriz_payoff[index_map[self.estrategia], index_map[viz]]
    self.fitness = payoff_total

def calcular_media_vizinhos(lagartos, estrategias, tipo):
    if tipo == 'interacao':
        medias_interacao = []
        for e in estrategias:
            #viz = [lag.n_vizinhos_realizado for lag in lagartos if lag.estrategia == e]
            viz = [lag.n_vizinhos_interacao for lag in lagartos if lag.estrategia == e]
            medias_interacao.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_interacao # retorna a média de vizinhos para cada estratégia

    elif tipo == 'aprendizado':
        medias_aprendizado = []
        for e in estrategias:
            viz = [lag.n_vizinhos_aprendizado for lag in lagartos if lag.estrategia == e]
            medias_aprendizado.append(np.mean(viz) if len(viz) > 0 else 0)
        return medias_aprendizado # retorna a média de vizinhos para cada estratégia

def criar_lagartos(L, LN):
    lista = []
    for i in range(L):
        for j in range(L):
            estrategia = np.random.choice(estrategias)
            n_vizinhos_interacao = 24
            n_vizinhos_aprendizado = LN
            lista.append(Lagarto(i, j, estrategia, 0, [], [], [], [], n_vizinhos_interacao, n_vizinhos_aprendizado))
    return lista

def calcular_freq(mat, estrategia):
    if estrategia == 'O':
        return np.sum(mat == 'O') / (L ** 2)
    elif estrategia == 'Y':
        return np.sum(mat == 'Y') / (L ** 2)
    elif estrategia == 'B':
        return np.sum(mat == 'B') / (L ** 2)
    
def atualizar_lagartos(lagartos): # função que atualiza as estratégias dos lagartos com base no fitness dos vizinhos
    novas_estrategias = {} # Dicionário para armazenar as novas estratégias
    mapa = {(l.i, l.j): l for l in lagartos} # dicionário para acessar lagartos pela posição

    for lagarto in lagartos:
        melhor_estrategia = lagarto.estrategia # inicia com a própria estratégia
        maior_fitness = lagarto.fitness # verifica o fitness do próprio lagarto
        
        # verifica o fitness dos vizinhos
        for (ni, nj) in lagarto.coord_vizinhos_aprendizado:
            vizinho = mapa[(ni, nj)] # usa o dicionário para achar o vizinho
            if vizinho.fitness > maior_fitness: # se o fitness do vizinho for maior que o maior fitness atual
                maior_fitness = vizinho.fitness # atualiza o maior fitness
                melhor_estrategia = vizinho.estrategia # atualiza a melhor estratégia
                
            elif vizinho.fitness == maior_fitness:
                a = np.random.rand()
                if a < 0.5:
                    maior_fitness = vizinho.fitness # atualiza o maior fitness
                    melhor_estrategia = vizinho.estrategia # atualiza a melhor estratégia
                    
                else:
                    pass
                # se houver empate de fitness ou for menor, mantém a estratégia atual (não muda)

            else:
                pass
            
        novas_estrategias[(lagarto.i, lagarto.j)] = melhor_estrategia # armazena a nova estratégia no dicionário
        
     # atualiza as estratégias de todos os lagartos simultaneamente
    for lagarto in lagartos:
        lagarto.estrategia = novas_estrategias[(lagarto.i, lagarto.j)]  # atualiza estratégia
        
    return lagartos

In [ ]:
# iniciando a simulação
def simulacao(a, LN, prob_mutacao = None, seed = None):
    resultados = []

    for pop in range(n_pop): # loop para cada população independente
        if seed is not None:
          np.random.seed(seed + pop) # coloca uma semente diferente pra cada pop, garantindo independência e reproducibilidade

        matriz_posicao = np.full((L, L), None) # cria uma matriz vazia com None
        lista_lagartos = criar_lagartos(L, LN) # cria os lagartos

        for lagarto in lista_lagartos:
            matriz_posicao[lagarto.i, lagarto.j] = str(lagarto.estrategia) # cria a matriz de posições de acordo com os lagartos

        #print(matriz_posicao)

        freq_O = calcular_freq(matriz_posicao, 'O') # calcula a frequência inicial
        freq_Y = calcular_freq(matriz_posicao, 'Y') # calcula a frequência inicial
        freq_B = calcular_freq(matriz_posicao, 'B') # calcula a frequência inicial

        resultados.append({
           "pop": pop,
           "t": 0,
           "freq_O": freq_O,
           "freq_Y": freq_Y,
           "freq_B": freq_B,
           "a": a,
           "IN": 24,
           "LN": LN
        })

        for t in range(1, n_geracoes + 1): # loop para cada geração dentro da população
          print(f"Vizinhanca LN {LN} - População {pop} - Geração {t}/{n_geracoes}")
          # determinando os vizinhos
          for lagarto in lista_lagartos:
            lagarto.calcular_coord_vizinhos(L, 'interacao') 
            lagarto.calcular_coord_vizinhos(L, 'aprendizado') 
            #print(f"Lagarto ({lagarto.i}, {lagarto.j}) - Estratégia: {lagarto.estrategia} - Vizinhos de interação: {len(lagarto.coord_vizinhos_interacao)} - Vizinhos de aprendizado: {len(lagarto.coord_vizinhos_aprendizado)}")
            lagarto.obter_estrategia_vizinhos(matriz_posicao) 
            lagarto.calcular_fitness(matriz_payoff, index_map) 
            #print(f"Lagarto ({lagarto.i}, {lagarto.j}) - Estratégia: {lagarto.estrategia} - Fitness: {lagarto.fitness}")

          lista_lagartos = atualizar_lagartos(lista_lagartos) # atualiza as estratégias dos lagartos de acordo com o maior fitness dos vizinhos

          if prob_mutacao is not None:
            for lagarto in lista_lagartos:
              lagarto.mutacao(prob_mutacao) # aplica a mutação

          matriz_posicao = np.full((L, L), None)
          
          for lagarto in lista_lagartos:
            matriz_posicao[lagarto.i, lagarto.j] = str(lagarto.estrategia)

          #print(matriz_posicao)

          freq_O = calcular_freq(matriz_posicao, 'O')
          freq_Y = calcular_freq(matriz_posicao, 'Y')
          freq_B = calcular_freq(matriz_posicao, 'B')
          
          resultados.append({
           "pop": pop,
           "t": t,
           "freq_O": freq_O,
           "freq_Y": freq_Y,
           "freq_B": freq_B,
           "a": a,
           "IN": 24,
           "LN": LN
        })
          
    return resultados

simulacao(a = a, LN = 8, prob_mutacao = prob_mutacao, seed = 1)

Vizinhanca LN 8 - População 0 - Geração 1/2
Lagarto (0, 0) - Estratégia: Y - Fitness: 11.0
Lagarto (0, 1) - Estratégia: O - Fitness: 7.5
Lagarto (0, 2) - Estratégia: O - Fitness: 7.5
Lagarto (0, 3) - Estratégia: Y - Fitness: 9.0
Lagarto (0, 4) - Estratégia: Y - Fitness: 8.5
Lagarto (1, 0) - Estratégia: O - Fitness: 7.5
Lagarto (1, 1) - Estratégia: O - Fitness: 7.5
Lagarto (1, 2) - Estratégia: Y - Fitness: 11.0
Lagarto (1, 3) - Estratégia: O - Fitness: 6.5
Lagarto (1, 4) - Estratégia: Y - Fitness: 11.5
Lagarto (2, 0) - Estratégia: O - Fitness: 9.0
Lagarto (2, 1) - Estratégia: B - Fitness: 9.5
Lagarto (2, 2) - Estratégia: Y - Fitness: 9.5
Lagarto (2, 3) - Estratégia: B - Fitness: 9.0
Lagarto (2, 4) - Estratégia: O - Fitness: 9.5
Lagarto (3, 0) - Estratégia: B - Fitness: 7.0
Lagarto (3, 1) - Estratégia: Y - Fitness: 8.0
Lagarto (3, 2) - Estratégia: B - Fitness: 10.0
Lagarto (3, 3) - Estratégia: O - Fitness: 11.0
Lagarto (3, 4) - Estratégia: O - Fitness: 13.0
Lagarto (4, 0) - Estratégia: B

[{'pop': 0,
  't': 0,
  'freq_O': 0.4,
  'freq_Y': 0.32,
  'freq_B': 0.28,
  'a': 0.5,
  'IN': 24,
  'LN': 8},
 {'pop': 0,
  't': 1,
  'freq_O': 0.4,
  'freq_Y': 0.6,
  'freq_B': 0.0,
  'a': 0.5,
  'IN': 24,
  'LN': 8},
 {'pop': 0,
  't': 2,
  'freq_O': 0.04,
  'freq_Y': 0.96,
  'freq_B': 0.0,
  'a': 0.5,
  'IN': 24,
  'LN': 8}]

In [ ]:
for LN in valores:
    df = simulacao(a, LN, L, matriz_payoff, index_map, n_pop, prob_mutacao = None, seed = 1)
    output_dir = f"C:/Unicamp/mestrado/simulacoes/main/RPS-neighborhood/outputs/vizinhanca_aprendizado-interacao/LN_igual/matriz_1-a/LN_{LN}/a_{a}/"
    os.makedirs(output_dir, exist_ok=True)
    df.to_csv(os.path.join(output_dir, f"resultados_vizinhanca_LN{LN}.csv"), index=False)